# 5G Network Intelligence ? runtime setup
Run on Colab first; Kaggle is the fallback. This notebook checks infrastructure only.
It does not train a GRU or claim a GPU allocation in advance.

**Colab:** Runtime ? Change runtime type ? select an available GPU.
**Kaggle:** Notebook Settings ? Accelerator ? GPU; enable Internet.
For a private GitHub repo, upload the source ZIP downloaded from GitHub into
`/content/5g-network-intelligence.zip` (Colab), or attach it as a private Kaggle input.
Never paste a GitHub token into a notebook cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/working').is_dir()
IS_COLAB = not IS_KAGGLE and 'COLAB_RELEASE_TAG' in os.environ
WORK = Path('/kaggle/working' if IS_KAGGLE else '/content')
# Set to the uploaded ZIP path, including /kaggle/input/... on Kaggle.
SOURCE_ZIP = WORK / '5g-network-intelligence.zip'
PROJECT = WORK / '5g-network-intelligence'
if not (PROJECT / 'pyproject.toml').exists():
    if not SOURCE_ZIP.exists():
        raise FileNotFoundError(f'Upload source ZIP and set SOURCE_ZIP: {SOURCE_ZIP}')
    extract_dir = WORK / 'source'
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        # Reject paths outside the selected extraction directory.
        for member in archive.infolist():
            if not (extract_dir / member.filename).resolve().is_relative_to(extract_dir.resolve()):
                raise ValueError('Unsafe ZIP member')
        archive.extractall(extract_dir)
    candidates = list(extract_dir.glob('*/pyproject.toml'))
    if len(candidates) != 1:
        raise ValueError('Expected exactly one project in the source ZIP')
    PROJECT = candidates[0].parent
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)


## Persistent checkpoints
Colab will ask for Drive authorization. On Kaggle, save a notebook version with
outputs before ending the session; /kaggle/working is not persistent across fresh sessions.


In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/5g-network-intelligence/checkpoints')
elif IS_KAGGLE:
    CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
else:
    raise RuntimeError('Use a Colab or Kaggle notebook runtime')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
probe = CHECKPOINT_DIR / '.write-probe'
probe.write_text('checkpoint storage writable')
assert probe.read_text() == 'checkpoint storage writable'
probe.unlink()
print('Checkpoint directory:', CHECKPOINT_DIR)


In [ ]:
from scripts.check_runtime import inspect_runtime
import json
runtime = inspect_runtime()
runtime['checkpoint_dir'] = str(CHECKPOINT_DIR)
(CHECKPOINT_DIR / 'runtime.json').write_text(json.dumps(runtime, indent=2))
print(json.dumps(runtime, indent=2))
if not runtime['cuda_available']:
    raise RuntimeError('GPU not available to PyTorch. Select a GPU runtime before Week 4 training.')


## Week 1 pipeline (CPU)
Optional: this downloads the public pinned dataset to the notebook runtime.
GPU epoch time, quota and training throughput remain unverified until the Week 4 pilot.


In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.prepare_data', '--step', 'all'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
